In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import json
import re
import networkx as nx
import pandas as pd
import plotly.graph_objects as go



from IPython.display import display, Markdown
from dotenv import load_dotenv
from openai import OpenAI, BadRequestError
from __future__ import annotations

from typing import Any, Iterable, Optional



load_dotenv()

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

print("✓ Imports complete")

✓ Imports complete


In [2]:
df_processed = pd.read_csv('../../data/FinalDataset/df_processed.csv', index_col=0)

In [3]:
def normalize_features(features: Iterable[str]) -> list[str]:
    """Clean and validate feature names."""
    cleaned = [str(feature).strip() for feature in features if str(feature).strip()]

    if not cleaned:
        raise ValueError("At least one non-empty feature name is required.")

    duplicates = sorted(
        feature for feature in set(cleaned) if cleaned.count(feature) > 1
    )
    if duplicates:
        raise ValueError(f"Duplicate feature names are not allowed: {duplicates}")

    return cleaned


def build_supply_chain_taxonomy_prompt(features: Iterable[str]) -> str:
    """Create the concise prompt used to generate the taxonomy."""
    feature_list = normalize_features(features)

    return f"""
You are a supply-chain data expert.

Given a list of dataset feature names, create a hierarchical taxonomy based on
the meaning and expressiveness of the feature names.

Rules:
1. Preserve every feature name exactly.
2. Include every feature in the taxonomy.
3. Create concise conceptual parent nodes only when needed.
4. Broader supply-chain concepts must appear at higher levels.
5. More specific measurements and engineered features must appear at lower levels.
6. Place lagged, rolling, aggregated, encoded, ratio, difference, and statistical
   features below their identifiable base feature.
7. Assign every child exactly one parent.
8. Do not create duplicate relationships, self-links, or cycles.
9. Do not infer relationships from correlation or causality.
10. Keep the taxonomy compact and readable.
11. `level` is the depth of the child node, with the root at level 0.
12. Every leaf must be one of the supplied dataset features.

Return valid JSON only in this exact structure:

{{
  "taxonomy": [
    {{
      "feature_name": "parent node",
      "child": "child node",
      "level": 1
    }}
  ]
}}

Dataset features:
{json.dumps(feature_list, ensure_ascii=False, indent=2)}
""".strip()

In [4]:
def response_to_text(response: Any) -> str:
    """Convert common LLM response objects to text."""
    if isinstance(response, str):
        return response

    output_text = getattr(response, "output_text", None)
    if isinstance(output_text, str):
        return output_text

    content = getattr(response, "content", None)
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and isinstance(item.get("text"), str):
                parts.append(item["text"])
            elif hasattr(item, "text"):
                parts.append(str(item.text))

        if parts:
            return "\n".join(parts)

    return str(response)


def invoke_llm(llm_model: Any, prompt: str) -> str:
    """
    Invoke an LLM object supporting one of these interfaces:

    - llm_model.invoke(prompt)
    - llm_model(prompt)
    - llm_model.generate(prompt)
    """
    if hasattr(llm_model, "invoke"):
        return response_to_text(llm_model.invoke(prompt))

    if callable(llm_model):
        return response_to_text(llm_model(prompt))

    if hasattr(llm_model, "generate"):
        return response_to_text(llm_model.generate(prompt))

    raise TypeError(
        "llm_model must be callable or expose invoke(prompt) or generate(prompt)."
    )


def extract_json_payload(response_text: str) -> dict[str, Any]:
    """Extract the taxonomy JSON object from the LLM response."""
    cleaned = response_text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        payload = json.loads(cleaned)
    except json.JSONDecodeError:
        start = cleaned.find("{")
        end = cleaned.rfind("}")

        if start == -1 or end == -1 or end <= start:
            raise ValueError("The LLM response does not contain valid JSON.")

        payload = json.loads(cleaned[start : end + 1])

    if not isinstance(payload, dict):
        raise ValueError("The LLM response must be a JSON object.")

    if "taxonomy" not in payload or not isinstance(payload["taxonomy"], list):
        raise ValueError('The JSON object must contain a list named "taxonomy".')

    return payload

In [5]:
class OpenRouterModel:
    """
    Callable OpenRouter chat model that satisfies the `llm_model(prompt)`
    interface expected by `invoke_llm`.

    Parameters
    ----------
    model:
        OpenRouter model slug, e.g. "openai/gpt-5.6-luna".

    max_tokens:
        Completion budget. The taxonomy JSON is long, so keep this generous.

    temperature:
        Left as None by default because some reasoning models only accept
        their default temperature.
    """

    OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

    def __init__(
        self,
        model: str,
        max_tokens: int = 16000,
        temperature: Optional[float] = None,
        timeout: float = 600.0,
    ) -> None:
        if not openrouter_api_key:
            raise RuntimeError(
                "OPENROUTER_API_KEY is not set. Add it to the .env file at the "
                "project root and re-run the imports cell."
            )

        self.model = model
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.client = OpenAI(
            base_url=self.OPENROUTER_BASE_URL,
            api_key=openrouter_api_key,
            timeout=timeout,
        )

    def _create(self, prompt: str, json_mode: bool) -> Any:
        request: dict[str, Any] = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": self.max_tokens,
        }

        if self.temperature is not None:
            request["temperature"] = self.temperature

        if json_mode:
            request["response_format"] = {"type": "json_object"}

        return self.client.chat.completions.create(**request)

    def __call__(self, prompt: str) -> str:
        try:
            response = self._create(prompt, json_mode=True)
        except BadRequestError:
            # Not every provider accepts response_format; the response parser
            # already tolerates fenced or prose-wrapped JSON.
            response = self._create(prompt, json_mode=False)

        message = response.choices[0].message
        content = (message.content or "").strip()

        if not content:
            finish_reason = getattr(response.choices[0], "finish_reason", None)
            raise ValueError(
                f"{self.model} returned an empty completion "
                f"(finish_reason={finish_reason!r}). Try raising max_tokens."
            )

        return content

    def __repr__(self) -> str:
        return f"OpenRouterModel(model={self.model!r})"

In [6]:
def validate_taxonomy_dataframe(taxonomy_df: pd.DataFrame, features: Iterable[str], ) -> pd.DataFrame:
    """Validate and standardize the generated taxonomy dataframe."""
    required_columns = ["feature_name", "child", "level"]
    missing = [column for column in required_columns if column not in taxonomy_df.columns]

    if missing:
        raise ValueError(f"Missing dataframe columns: {missing}")

    df = taxonomy_df[required_columns].copy()

    if df.empty:
        raise ValueError("The taxonomy contains no relationships.")

    df["feature_name"] = df["feature_name"].astype(str).str.strip()
    df["child"] = df["child"].astype(str).str.strip()
    df["level"] = pd.to_numeric(df["level"], errors="raise").astype(int)

    if (df["feature_name"] == "").any() or (df["child"] == "").any():
        raise ValueError("Parent and child names must not be empty.")

    if (df["feature_name"] == df["child"]).any():
        raise ValueError("Self-links are not allowed.")

    if (df["level"] < 1).any():
        raise ValueError("Every child represented by an edge must have level >= 1.")

    df = df.drop_duplicates().reset_index(drop=True)

    duplicate_children = df[df["child"].duplicated(keep=False)]
    if not duplicate_children.empty:
        conflicts = (
            duplicate_children.groupby("child")["feature_name"].apply(list).to_dict()
        )
        raise ValueError(f"Every child must have one parent. Conflicts: {conflicts}")

    graph = nx.DiGraph()
    graph.add_edges_from(
        df[["feature_name", "child"]].itertuples(index=False, name=None)
    )

    if not nx.is_directed_acyclic_graph(graph):
        raise ValueError("The generated taxonomy contains a cycle.")

    supplied_features = set(normalize_features(features))
    omitted_features = sorted(supplied_features - set(graph.nodes))

    if omitted_features:
        raise ValueError(
            f"The following supplied features were omitted: {omitted_features}"
        )

    roots = [node for node in graph.nodes if graph.in_degree(node) == 0]
    if not roots:
        raise ValueError("The generated taxonomy contains no root node.")

    expected_levels: dict[str, int] = {}
    for root in roots:
        expected_levels[root] = 0
        for parent, child in nx.bfs_edges(graph, root):
            expected_levels[child] = expected_levels[parent] + 1

    invalid_levels = []
    for row in df.itertuples(index=False):
        expected = expected_levels[row.child]
        if row.level != expected:
            invalid_levels.append(
                {
                    "feature_name": row.feature_name,
                    "child": row.child,
                    "provided_level": row.level,
                    "expected_level": expected,
                }
            )

    if invalid_levels:
        raise ValueError(f"Inconsistent hierarchy levels: {invalid_levels}")

    leaf_nodes = {node for node in graph.nodes if graph.out_degree(node) == 0}
    conceptual_leaves = sorted(leaf_nodes - supplied_features)

    if conceptual_leaves:
        raise ValueError(
            "Every leaf must be a supplied feature. "
            f"Invalid conceptual leaves: {conceptual_leaves}"
        )

    return (
        df.sort_values(["level", "feature_name", "child"])
        .reset_index(drop=True)
    )


def generate_supply_chain_taxonomy(
    llm_model: Any,
    features: Iterable[str],
    max_retries: int = 2,
) -> pd.DataFrame:
    """
    Generate a validated supply-chain taxonomy dataframe.

    Parameters
    ----------
    llm_model:
        LLM object supporting invoke(prompt), direct calling, or generate(prompt).

    features:
        Exact dataset feature names.

    max_retries:
        Number of correction attempts after an invalid LLM response.
    """
    feature_list = normalize_features(features)
    original_prompt = build_supply_chain_taxonomy_prompt(feature_list)
    current_prompt = original_prompt
    last_error: Optional[Exception] = None

    for attempt in range(max_retries + 1):
        response_text = invoke_llm(llm_model, current_prompt)

        try:
            payload = extract_json_payload(response_text)
            taxonomy_df = pd.DataFrame(payload["taxonomy"])
            return validate_taxonomy_dataframe(taxonomy_df, feature_list)

        except Exception as error:
            last_error = error

            if attempt == max_retries:
                break

            current_prompt = f"""
Your previous response was invalid.

Validation error:
{type(error).__name__}: {error}

Regenerate the complete taxonomy from scratch.
Preserve every supplied feature name exactly.
Include every supplied feature.
Return valid JSON only.

Original task:
{original_prompt}
""".strip()

    raise ValueError(
        f"The LLM failed to create a valid taxonomy after {max_retries + 1} attempts."
    ) from last_error

In [7]:
def taxonomy_icon(level: int, is_root: bool = False) -> str:
    """
    Assign an icon strictly according to hierarchy level.

    Level 0: root
    Level 1: broad domain
    Level 2: subdomain or process
    Level 3: concept or KPI family
    Level 4: specific measurement
    Level 5+: engineered or highly detailed feature
    """
    if is_root or level == 0:
        return "🌐"

    level_icons = {
        1: "🗂️",
        2: "📁",
        3: "📌",
        4: "🔹",
        5: "⚙️",
    }

    return level_icons.get(level, "▫️")


def calculate_taxonomy_positions(
    graph: nx.DiGraph,
) -> dict[str, tuple[float, float]]:
    """Calculate a clean left-to-right hierarchical layout."""
    roots = sorted(node for node in graph.nodes if graph.in_degree(node) == 0)

    if not roots:
        raise ValueError("The taxonomy contains no root.")

    positions: dict[str, tuple[float, float]] = {}
    next_y = 0.0

    def place_subtree(node: str, depth: int) -> float:
        nonlocal next_y

        children = sorted(graph.successors(node))

        if not children:
            y = next_y
            next_y += 1.0
        else:
            child_positions = [place_subtree(child, depth + 1) for child in children]
            y = sum(child_positions) / len(child_positions)

        positions[node] = (float(depth), y)
        return y

    for root in roots:
        place_subtree(root, 0)
        next_y += 1.0

    return positions


def visualize_supply_chain_taxonomy(
    taxonomy_df: pd.DataFrame,
    title: str = "Supply-Chain Feature Taxonomy",
    output_html: Optional[str] = None,
    show: bool = True,
) -> go.Figure:
    """Display an interactive taxonomy with hierarchy-aware icons."""
    required_columns = {"feature_name", "child", "level"}
    missing = required_columns - set(taxonomy_df.columns)

    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    df = taxonomy_df[["feature_name", "child", "level"]].copy()
    df["feature_name"] = df["feature_name"].astype(str).str.strip()
    df["child"] = df["child"].astype(str).str.strip()
    df["level"] = pd.to_numeric(df["level"], errors="raise").astype(int)

    graph = nx.DiGraph()
    graph.add_edges_from(
        df[["feature_name", "child"]].itertuples(index=False, name=None)
    )

    if not nx.is_directed_acyclic_graph(graph):
        raise ValueError("The taxonomy contains a cycle.")

    positions = calculate_taxonomy_positions(graph)
    roots = {node for node in graph.nodes if graph.in_degree(node) == 0}

    edge_x, edge_y = [], []

    for parent, child in graph.edges:
        parent_x, parent_y = positions[parent]
        child_x, child_y = positions[child]
        elbow_x = parent_x + 0.45 * (child_x - parent_x)

        edge_x.extend([parent_x, elbow_x, elbow_x, child_x, None])
        edge_y.extend([parent_y, parent_y, child_y, child_y, None])

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line={"width": 1.5, "color": "rgba(82, 96, 115, 0.55)"},
        hoverinfo="skip",
    )

    ordered_nodes = sorted(
        graph.nodes,
        key=lambda node: (positions[node][0], positions[node][1], node),
    )

    node_x = [positions[node][0] for node in ordered_nodes]
    node_y = [positions[node][1] for node in ordered_nodes]
    node_levels = [int(positions[node][0]) for node in ordered_nodes]

    node_sizes = [
        30 if node in roots else 22 if graph.out_degree(node) > 0 else 16
        for node in ordered_nodes
    ]

    node_labels = [
        f"{taxonomy_icon(int(positions[node][0]), node in roots)} {node}"
        for node in ordered_nodes
    ]

    hover_labels = [
        (
            f"<b>{node}</b>"
            f"<br>Level: {int(positions[node][0])}"
            f"<br>Parent: {next(iter(graph.predecessors(node)), 'Root')}"
            f"<br>Children: {graph.out_degree(node)}"
        )
        for node in ordered_nodes
    ]

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=node_labels,
        textposition="middle right",
        hovertext=hover_labels,
        hoverinfo="text",
        marker={
            "size": node_sizes,
            "color": node_levels,
            "colorscale": "Blues",
            "showscale": True,
            "colorbar": {"title": "Level", "thickness": 14},
            "line": {"width": 1.5, "color": "white"},
        },
        textfont={
            "size": 12,
            "family": (
                "Arial, Segoe UI Emoji, Apple Color Emoji, "
                "Noto Color Emoji, sans-serif"
            ),
        },
    )

    maximum_depth = max(node_levels)
    leaf_count = sum(1 for node in graph.nodes if graph.out_degree(node) == 0)

    figure = go.Figure(data=[edge_trace, node_trace])

    figure.update_layout(
        title={"text": title, "x": 0.02, "xanchor": "left"},
        template="plotly_white",
        width=max(1050, 270 * (maximum_depth + 1)),
        height=max(550, 37 * leaf_count + 180),
        margin={"l": 40, "r": 320, "t": 90, "b": 50},
        xaxis={
            "title": "Level of abstraction",
            "tickmode": "linear",
            "dtick": 1,
            "showgrid": True,
            "gridcolor": "rgba(160, 170, 180, 0.20)",
            "zeroline": False,
        },
        yaxis={"visible": False, "autorange": "reversed"},
        showlegend=False,
        plot_bgcolor="white",
        paper_bgcolor="white",
    )

    if output_html:
        figure.write_html(output_html, include_plotlyjs="cdn")

    if show:
        figure.show()

    return figure

## Provide your features and LLM model

`llm_model` must be an object, not a bare model-id string. `OpenRouterModel`
wraps an OpenRouter slug into a callable that satisfies one of the interfaces
`invoke_llm` accepts:

- `llm_model.invoke(prompt)`
- `llm_model(prompt)`
- `llm_model.generate(prompt)`

It reads `OPENROUTER_API_KEY` from the `.env` file at the project root.

In [17]:
# Dataset feature names.
features = df_processed.columns.to_list()

# Available OpenRouter model slugs.
GPT = "openai/gpt-5.6-luna"
GEMENI = "google/gemini-3.5-flash-lite"
MISTRAL = "mistralai/mistral-small-2603"
ANTHROPIC = "anthropic/claude-haiku-4.5"

# Wrap the slug in a callable client so `invoke_llm` can call it.
llm_model = OpenRouterModel(ANTHROPIC)

print(f"{len(features)} features | {llm_model!r}")

129 features | OpenRouterModel(model='anthropic/claude-haiku-4.5')


In [18]:
# Generate the taxonomy dataframe.
taxonomy_df = generate_supply_chain_taxonomy(
    llm_model=llm_model,
    features=features,
    max_retries=2,
)

display(taxonomy_df)

,feature_name,child,level
0,Calendar Events,category_bank_holiday,1
1,Calendar Events,category_cultural_event,1
2,Calendar Events,category_cultural_holiday,1
3,Calendar Events,category_national_holiday,1
4,Calendar Events,category_no_event,1
...,...,...,...
124,Sales,Sales_lag_2,2
125,Sales,Sales_lag_21,2
126,Sales,Sales_lag_28,2
127,Sales,Sales_lag_3,2


In [ ]:
# Save the dataframe. The slug contains "/", so flatten it into a filename.
output_dir = "../../taxonomy"
os.makedirs(output_dir, exist_ok=True)

model_slug = re.sub(r"[^A-Za-z0-9._-]+", "_", llm_model.model)
taxonomy_df.to_csv(
    f"{output_dir}/supply_chain_taxonomy_{model_slug}.csv", index=False
)

# Display and optionally save the interactive visualization.
taxonomy_figure = visualize_supply_chain_taxonomy(
    taxonomy_df=taxonomy_df,
    title="Supply-Chain Feature Taxonomy",
    output_html=f"{output_dir}/supply_chain_taxonomy_{model_slug}.html",
    show=True,
)

The models were selected to evaluate whether compact language models can generate coherent supply-chain feature taxonomies while maintaining low computational cost and reliable structured output. This task requires the model to interpret the semantic meaning of short feature names, identify broader–narrower relationships, recognize engineered variables such as lagged and rolling features, and return the resulting hierarchy in a machine-readable format.

The use of language models for this task is supported by research showing that pretrained language models encode lexical-semantic relations such as hypernymy and can use these relations for taxonomy construction and expansion. TaxoLLaMA, for example, achieved state-of-the-art or near-state-of-the-art results across taxonomy construction, taxonomy enrichment, hypernym discovery, and lexical-entailment tasks. Similarly, zero-shot taxonomy-induction research has demonstrated that hierarchical relations can be extracted from language models through prompting. More recent studies also show that LLM-based methods can produce taxonomies with improved coherence, granularity, and interpretability, while LLM-guided refinement can improve the effectiveness of hierarchical classification.

**GPT-5.6 Luna** was selected as the strong compact commercial baseline. It represents the lowest-cost tier of the GPT-5.6 family and is explicitly intended for cost-sensitive, high-volume workloads. Its inclusion allows the study to examine whether a relatively economical frontier-model variant can provide sufficient semantic understanding and instruction following for taxonomy generation.

**Gemini 3.5 Flash-Lite** was selected as a speed- and cost-oriented model. It is designed for high-throughput processing, document parsing, and simple data-extraction tasks, and it supports schema-constrained structured outputs. These characteristics are relevant because the proposed taxonomy procedure resembles a combination of semantic classification, hierarchical organization, and structured information extraction.

**Mistral Small 4** was selected as the open-weight alternative. It uses a mixture-of-experts architecture with 119 billion total parameters but only 6.5 billion active parameters, supports structured output, and is distributed under the Apache 2.0 licence. Its inclusion improves reproducibility and makes local or privately hosted deployment possible, which may be important for industrial supply-chain datasets containing commercially sensitive information. Scientific evidence also indicates that smaller models can be competitive for well-defined classification tasks and that, when appropriately adapted, small models can outperform larger frozen models in taxonomy-expansion settings.

The three models therefore represent complementary deployment configurations: a strong compact proprietary model, a high-throughput low-cost model, and an open-weight model suitable for reproducible or local deployment. Using models from different providers also makes it possible to determine whether the generated taxonomy is stable across model families rather than being an artefact of one provider’s training procedure.

All models should be used with native JSON-schema or structured-output functionality. Research on constrained generation shows that ordinary prompting alone does not guarantee syntactically valid structured results, whereas schema-constrained decoding substantially improves format compliance and parsability. However, constrained generation can still affect semantic quality, so the generated taxonomies should additionally be validated for missing features, duplicate parents, incorrect levels, and cycles.
